In [ ]:
import requests
import os
import pandas as pd
import time
from datetime import datetime, timedelta
import gzip
import shutil
import smtplib
from email.mime.text import MIMEText

base_url = os.environ['CD2_BASE_URL']
client_id = os.environ['CD2_CLIENT_ID']
client_secret = os.environ['CD2_CLIENT_SECRET']

In [ ]:
def send_epost(tittel, innhald, avsendar, mottakarar):
    msg = MIMEText(innhald)
    msg['Subject'] = tittel
    msg['From'] = avsendar
    msg['To'] = ', '.join(mottakarar)
    smtp_server = smtplib.SMTP('smtp-ut.hvl.no', port=25)
    smtp_server.sendmail(avsendar, mottakarar, msg.as_string())
    smtp_server.quit() 


def hent_filar_web_logs(innfil, n):
    """
    Hentar datafil frå CD2 og pakkar den ut.
    Argument:
    tabell - namnet på tabellen som skal hentast
    innfil - id på fila som skal hentast
    n - rekkjefølgje på fila som skal hentast (for logging)
    Returnerer namnet på den utpakkede fila.
    """
    requesturl = f"{base_url}/dap/object/url"
    payload = f"{respons2['objects']}"
    payload = payload.replace('\'', '\"')
    headers = {
        'x-instauth': access_token, 
        'Content-Type': 'text/plain'
        }
    print(f"Hentar datafil nr. {n}, ", end="")
    r4 = requests.post(
        requesturl, 
        headers=headers, 
        data=payload
        )
    if r4.status_code == 200:
        respons4 = r4.json()
        url = respons4['urls'][innfil]['url']
        data = requests.request("GET", url)
        no = datetime.now()
        utfil = f"web_logs-{no.year}{no.month:02}{no.day:02}{no.hour:02}{no.minute:02}-{n}"
        open(f'{utfil}.gz', 'wb').write(data.content)
        with gzip.open(f'{utfil}.gz', 'rb') as f_in:
            with open(f'{utfil}.txt', 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print(f" skrevet til {f'{utfil}.txt'}")
        os.remove(f"{utfil}.gz")
    return f"{utfil}.txt"

In [ ]:
auth_url = f"{base_url}/ids/auth/login"
payload={'grant_type': 'client_credentials'}
r = requests.post(
    auth_url, 
    data=payload, 
    auth=(client_id, client_secret))
if 200 <= r.status_code < 300:
    respons = r.json()
    access_token = respons['access_token']
    print("Henta access_token OK")
else:
    print(f"Klarte ikkje å skaffe access_token, feil {r.status_code}")

In [ ]:
no = datetime.now()
timar = 4
tidsrom = timedelta(hours=timar)
sist_oppdatert = (no - tidsrom).isoformat(timespec='seconds') + "Z"
# with open(f"sist_oppdatert_{tabell}.txt", "r") as f_in:
#     sist_oppdatert = f_in.read()
requesturl = f"{base_url}/dap/query/canvas_logs/table/web_logs/data"
payload = '{"format": "csv", "since": \"%s\"}' %(sist_oppdatert)
headers = {'x-instauth': access_token, 'Content-Type': 'text/plain'}
try:
    print(f"Sender inkrementell spørjing for dei siste {timar} timane til {requesturl}")
    r = requests.post(
        requesturl, 
        headers=headers, 
        data=payload
        )
    if 200 <= r.status_code < 300:
        respons = r.json()
        id = respons['id']
        les_data = True
        while les_data:
            print(f"Sjekker status på jobb {id}")
            requesturl = f"{base_url}/dap//job/{id}"
            r2 = requests.get(requesturl, headers=headers)
            if 200 <= r2.status_code < 300:
                respons2 = r2.json()
                print(respons2)
                if respons2['status'] == "complete":
                    les_data = False
                time.sleep(5)
        antal = len(respons2['objects'])
        filer_i_dag = []
        for i in range(antal):
            utfil = hent_filar_web_logs(respons2['objects'][i]['id'], i)
            filer_i_dag.append(utfil)
    else:
        print(f"Feil i spørjing, status {r.status_code}")
except Exception as e:
    print(f"Noko gjekk gale: {e}")

In [ ]:
urls = pd.read_csv(utfil, sep=',', usecols=['value.user_id', 'value.url'])
reelle_urls = urls[~urls['value.user_id'].isin([2916, 12477])]
digitalundervising = reelle_urls[reelle_urls['value.url'].str.contains('corses/12436/pages/')]

In [ ]:
avsendar = "aasmund.kvamme@hvl.no"
mottakarar = ["aasmund.kvamme@hvl.no"]
tittel = "Sidevisingar i Digitalundervising"
innhald = f"Sidevisingar i emnet Digitalundervising mellom kl. {sist_oppdatert[11:16]} og {no.isoformat()[11:16]} i dag: {len(digitalundervising)}\n\n"
for i, r in digitalundervising.iterrows():
    innhald += f"{r['value.user_id']}, {r['value.url']}\n"

if len(digitalundervising) > 0:
    send_epost(tittel, innhald, avsendar, mottakarar)